In [1]:
import pandas as pd
import torch
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from models import MLP

In [2]:
SEED: int = 42

## 1 (c)

Question: We have provided a dataset from taxi trips taken in NYC in taxi trips.parquet. It contains 8 total columns, 7 features and 1 target (trip duration, in minutes). The 7 features (pickup/dropoff coordinates and time-of-day, day-of-week and day-of-year) have already been standardized, so do not expect them to look like raw latitudes and longitudes. Define a multi-layer perceptron in PyTorch or JAX that outputs a conditional mean and standard deviation, and fit this on the taxi trip dataset. Make sure to follow best practices: define a baseline (or multiple) baseline models that you think your model should beat, split your dataset into training/validation/test sets, etc.

In [3]:
taxi_trips = pd.read_parquet('../data/taxi_trips.parquet')

taxi_trips.sample(n = 3)

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,min_of_day,day_of_week,day_of_year
155984,0.479161,0.641301,1.018946,0.676734,8.531511,0.671319,-1.557377,-1.008916
14901,-0.267593,1.067919,0.014536,0.817796,3.200330,-0.876579,0.481657,1.103341
240408,-1.362003,-1.997124,-0.867840,-1.475201,10.730246,0.137019,-0.537860,1.607182


In [18]:
taxi_trips.info()

<class 'pandas.core.frame.DataFrame'>
Index: 523554 entries, 187173 to 190792
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   pickup_longitude   523554 non-null  float32
 1   pickup_latitude    523554 non-null  float32
 2   dropoff_longitude  523554 non-null  float32
 3   dropoff_latitude   523554 non-null  float32
 4   trip_duration      523554 non-null  float32
 5   min_of_day         523554 non-null  float32
 6   day_of_week        523554 non-null  float32
 7   day_of_year        523554 non-null  float32
dtypes: float32(8)
memory usage: 20.0 MB


In [4]:
y_col  = 'trip_duration'
x_cols = [x for x in taxi_trips.columns if x != y_col]

X = taxi_trips[x_cols].values
y = taxi_trips[y_col].values

In [5]:
X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size = 0.30, random_state = SEED)
X_val, X_te, y_val, y_te   = train_test_split(X_temp, y_temp, test_size = 2/3, random_state = SEED)

scaler = StandardScaler()
y_tr  = scaler.fit_transform(y_tr.reshape(-1, 1)).ravel()
y_val = scaler.transform(y_val.reshape(-1, 1)).ravel()
y_te  = scaler.transform(y_te.reshape(-1, 1)).ravel()

n = len(X)

print(f"Train size: {len(X_tr) / n * 100:.2f}%\nValidation size: {len(X_val) / n * 100:.2f}%\nTest size: {len(X_te) / n * 100:.2f}%")

Train size: 70.00%
Validation size: 10.00%
Test size: 20.00%


In [6]:
# Baseline Model: Linear Gaussian Model

mlp_baseline = MLP(
    d_in = X_tr.shape[1],
    d_hidden = 128,
    activation_type = None,
    bias = True,
    batch_size = 256,
    epochs = 100,
    learning_rate = 1e-3,
    device = 'cpu',
    seed = SEED
)

mlp_baseline.load_state_dict(torch.load('saved_models/mlp_baseline.pt'))
mlp_baseline.eval()

MLP(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): Identity()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [7]:
# Parameter selection

param_selection_df = pd.read_csv('saved_models/results.csv')

param_selection_df.sort_values(by = 'loss', ascending = True).head()

,d_hidden,activation_type,learning_rate,loss
0,32,relu,0.010,-0.253930
3,32,tanh,0.010,-0.247423
13,256,tanh,0.001,-0.214033
22,1024,tanh,0.001,-0.210022
6,32,silu,0.010,-0.203315


In [8]:
# My Model: Non-Linear Gaussian Model

mlp_best = MLP(
    d_in = X_tr.shape[1],
    d_hidden = 32,
    activation_type = 'relu',
    bias = True,
    batch_size = 256,
    epochs = 100,
    learning_rate = 1e-2,
    device = 'cpu',
    seed = SEED
)

mlp_best.load_state_dict(torch.load('saved_models/mlp_32_relu_0.01.pt'))
mlp_best.eval()

MLP(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
)

In [9]:
best_pred = mlp_best.predict(X_te)
baseline_pred = mlp_baseline.predict(X_te)

best_pred_t = torch.from_numpy(best_pred).float()
baseline_pred_t = torch.from_numpy(baseline_pred).float()
y_t = torch.from_numpy(y_te).float().view(-1, 1)

print(f"Best MLP Test Loss: {mlp_best.loss_func(best_pred_t, y_t).item():.4f}")           
print(f"Baseline MLP Test Loss: {mlp_baseline.loss_func(baseline_pred_t, y_t).item():.4f}")

Best MLP Test Loss: -0.1805
Baseline MLP Test Loss: 0.4172


## 1 (d)

Question: How would we use the model we’ve just fit to predict the probability that a trip will take less than 45 minutes? The normal distribution assumption is fine?

Response: We can use a normal CDF parameterized with the values we found from our neural network! We solve: $$\mathbb{P}[Y \le 45 | X = x] = \Phi(\frac{45 - f_{\mu}(x)}{f_{\sigma}(x)})$$

where:
- $\Phi(\cdot)$ is the standard normal cdf
- $f_{\mu}(x) = g_1(x) \cdot s + \bar{Y}$
- $f_{\sigma}(x) = \exp{(g_2(x))} \cdot s$
- $g(x) = (g_1(x), g_2(x))$ is the network's output as defined in the problem
- $\bar{Y}$ is the training mean of $y_i$ and $s$ is the training standard deviation of $y_i$

## 1 (e)

Estimate these probabilities on your held-out test set and evaluate them. Report both a proper scoring rule for the predicted probabilities (e.g. the Brier score or the log-loss) and the classification accuracy obtained by thresholding the probability at 0.5.

In [10]:
mu_std = best_pred[:, 0]
sigma_std = np.exp(np.clip(best_pred[:, 1], -7.0, 5.0))

mu = scaler.inverse_transform(mu_std.reshape(-1, 1)).ravel()
sigma = sigma_std * scaler.scale_[0]

best_mlp_probabilities = stats.norm.cdf(45, loc = mu, scale = sigma)

In [11]:
mu_std = baseline_pred[:, 0]
sigma_std = np.exp(np.clip(baseline_pred[:, 1], -7.0, 5.0))

mu = scaler.inverse_transform(mu_std.reshape(-1, 1)).ravel()
sigma = sigma_std * scaler.scale_[0]

baseline_mlp_probabilities = stats.norm.cdf(45, loc = mu, scale = sigma)

In [12]:
observations = (scaler.inverse_transform(y_te.reshape(-1, 1)).ravel() < 45).astype(int)

In [13]:
base_rate = observations.mean()
brier_score_base_rate = base_rate * (1 - base_rate)

In [14]:
best_mlp_brier_score = np.mean((best_mlp_probabilities - observations) ** 2)
baseline_mlp_brier_score = np.mean((baseline_mlp_probabilities - observations) ** 2)

print(f"Brier score for Best MLP: {best_mlp_brier_score:.6f}")
print(f"Brier score for Baseline MLP: {baseline_mlp_brier_score:.6f}")
print(f"Brier score for Base Rate: {brier_score_base_rate:.6f}")

Brier score for Best MLP: 0.003581
Brier score for Baseline MLP: 0.004424
Brier score for Base Rate: 0.004421


In [15]:
preds = (best_mlp_probabilities >= 0.5).astype(int)
best_mlp_accuracy = (preds == observations).mean()

In [16]:
preds = (baseline_mlp_probabilities >= 0.5).astype(int)
baseline_mlp_accuracy = (preds == observations).mean()

In [ ]:
print(f"Classification Accuracy for Best MLP: {best_mlp_accuracy*100:.4f}%")
print(f"Classification Accuracy for Baseline MLP: {baseline_mlp_accuracy*100:.4f}%")

Accuracy for Best MLP: 99.5464%
Accuracy for Baseline MLP: 99.5559%


### Comment:

The Best MLP has a Brier score of 0.003581, the Baseline MLP has a Brier score of 0.004424, and the base rate Brier score is 0.004421. Given how close the Brier scores are for the Baseline MLP and the base rate, we posit that the Baseline MLP has no real predictive skill. However, the Best MLP, which has a non-linearity, modestly enhances predictive power.

The Best MLP has an accuracy of 99.55%, and the Baseline MLP has an accuracy of 99.56%. In the context of the data and our problem, accuracy may not be a meaningful metric because the Baseline MLP acts as a trivial classifier, always predicting that a trip lasts less than 45 minutes.